In [34]:
import numpy as np
import pandas as pd

df_wdi_raw = pd.read_excel("WDI.xlsx")
df_edu_raw = pd.read_excel("World_Edu.xls")

In [35]:
# Reshape WDI from long to wide format and rename indicators to readable feature names
df_wdi = df_wdi_raw.copy()
df_wdi["2022 [YR2022]"] = pd.to_numeric(df_wdi["2022 [YR2022]"], errors="coerce")

df_wdi_wide = df_wdi.pivot_table(
    index=["Country Name", "Country Code"],
    columns="Series Code",
    values="2022 [YR2022]",
    aggfunc="first"
).reset_index()

rename_dict = {
    "Country Code":       "iso3",
    "NY.GDP.PCAP.PP.CD":  "gdp_pc_ppp",
    "SI.POV.GINI":        "gini_index",
    "SE.XPD.TOTL.GD.ZS":  "gov_edu_spend",
    "SP.URB.TOTL.IN.ZS":  "urban_pct",
    "SP.POP.0014.TO.ZS":  "pop_0_14",
    "SP.POP.1564.TO.ZS":  "pop_15_64",
    "SP.POP.65UP.TO.ZS":  "pop_65_plus",
    "SL.TLF.CACT.FE.ZS":  "lfp_female",
    "SL.TLF.CACT.MA.ZS":  "lfp_male",
}
df_wdi_wide = df_wdi_wide.rename(columns=rename_dict)
keep_cols = list(rename_dict.values()) + ["Country Name"]
df_wdi_wide = df_wdi_wide[[c for c in keep_cols if c in df_wdi_wide.columns]]

In [36]:
# Calculate youth dependency ratio (children per working-age adult).
# We use this instead of total dependency because the elderly population 
# doesn't directly impact education demand or policy.
df_wdi_wide["youth_dependency_ratio"] = df_wdi_wide["pop_0_14"] / df_wdi_wide["pop_15_64"]

# Calculate the labor force participation gender gap. 
# This serves as a proxy for the expected economic return on female education.
df_wdi_wide["lfp_gender_gap"] = df_wdi_wide["lfp_male"] - df_wdi_wide["lfp_female"]

df_wdi_features = df_wdi_wide.drop(
    columns=["pop_0_14", "pop_15_64", "pop_65_plus", "lfp_male", "lfp_female"]
)

In [38]:
df_edu = df_edu_raw.copy()
df_edu["iso3"] = df_edu["iso3"].str.strip().str.upper()
df_wdi_features["iso3"] = df_wdi_features["iso3"].str.strip().str.upper()

# Select one variable per policy domain from the WORLD codebook:
#   finbar_prim:   Financial access (tuition-free primary education)
#   compend_prim:  Legal mandate (compulsory primary education)
#   disc_sex_prim: Gender anti-discrimination protections
#   incl_edu:      Disability inclusion guarantees

# Note: Dropped 'sh_edu' because it heavily overlaps with 'disc_sex_prim'. 
# Keeping both would introduce redundancy in the model.
policy_cols = ["iso3", "country", "region", "wb_econ", "finbar_prim", "compend_prim", "disc_sex_prim", "incl_edu"]
df_edu_clean = df_edu[policy_cols]

In [39]:
# Use an inner join to keep only countries that have both macro and policy data
df_master = pd.merge(df_edu_clean, df_wdi_features, on="iso3", how="inner")
df_master = df_master.drop(columns=["Country Name"], errors="ignore")
df_master = df_master.replace(r"^\s*$", np.nan, regex=True)

In [40]:
# Check for missing values before finalizing our feature set
all_candidate_features = ["gdp_pc_ppp", "urban_pct", "youth_dependency_ratio", "lfp_gender_gap", "finbar_prim", "compend_prim", "disc_sex_prim", "incl_edu", "gini_index", "gov_edu_spend"]
available = [c for c in all_candidate_features if c in df_master.columns]
n_total = len(df_master)
nan_audit = pd.DataFrame({
    "missing_n":   df_master[available].isna().sum(), 
    "missing_pct": (df_master[available].isna().sum() / n_total * 100).round(1)
}).sort_values("missing_pct", ascending=False)
print(f"Total countries after merge: {n_total}\n")
print(nan_audit)

Total countries after merge: 193

                        missing_n  missing_pct
gini_index                    121         62.7
gov_edu_spend                  38         19.7
lfp_gender_gap                 16          8.3
gdp_pc_ppp                      8          4.1
incl_edu                        5          2.6
finbar_prim                     1          0.5
urban_pct                       0          0.0
youth_dependency_ratio          0          0.0
compend_prim                    0          0.0
disc_sex_prim                   0          0.0


In [41]:
# Since PCA and clustering drop rows with missing values (listwise deletion),
# We check how many countries we lose under different feature scenarios.
core_features = ["gdp_pc_ppp", "urban_pct", "youth_dependency_ratio", "lfp_gender_gap", "finbar_prim", "compend_prim", "disc_sex_prim", "incl_edu"]
scenarios = {
    "Core only":            core_features,
    "Core + gov_edu_spend": core_features + ["gov_edu_spend"],
    "Core + gini_index":    core_features + ["gini_index"],
    "Core + both":          core_features + ["gov_edu_spend", "gini_index"],
}
for label, cols in scenarios.items():
    available_cols = [c for c in cols if c in df_master.columns]
    n = df_master.dropna(subset=available_cols).shape[0]
    print(f"{label: <30} {n} countries retained")

Core only                      165 countries retained
Core + gov_edu_spend           136 countries retained
Core + gini_index              72 countries retained
Core + both                    64 countries retained


In [42]:
# Each of the four WORLD policy variables captures a distinct area:
# financial access, compulsory mandate, gender protection, and disability inclusion.
# 
# These variables are ordinal but have non-uniform spacing. For instance,
# finbar_prim, compend_prim, and disc_sex_prim use {1, 3, 4, 5}, skipping 2.
# We remap them to consecutive integers (0-3) to remove artificial gaps 
# while keeping their original order intact.

finbar_prim_map = {1: 0, 3: 1, 4: 2, 5: 3}
# 0: not tuition-free
# 1: subject to progressive realization
# 2: policy guarantee
# 3: legislative or constitutional guarantee

compend_prim_map = {1: 0, 3: 1, 4: 2, 5: 3}
# 0: not compulsory
# 1: subject to progressive realization
# 2: policy guarantee
# 3: legislative or constitutional guarantee

disc_sex_prim_map = {1: 0, 3: 1, 4: 2, 5: 3}
# 0: no prohibition
# 1: broadly prohibited, not education-specific
# 2: prohibited in admissions or access
# 3: prohibited in education

incl_edu_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
# 0: no guarantee
# 1: aspirational or subject to progressive realization
# 2: preferential placement in inclusive environments
# 3: guaranteed right with exceptions
# 4: guaranteed right without exceptions

df_master["finbar_prim"]   = df_master["finbar_prim"].map(finbar_prim_map)
df_master["compend_prim"]  = df_master["compend_prim"].map(compend_prim_map)
df_master["disc_sex_prim"] = df_master["disc_sex_prim"].map(disc_sex_prim_map)
df_master["incl_edu"]      = df_master["incl_edu"].map(incl_edu_map)

print("Encoded value distributions:")
for col in ["finbar_prim", "compend_prim", "disc_sex_prim", "incl_edu"]:
    print(f"  {col}: {dict(sorted(df_master[col].value_counts().items()))}")

Encoded value distributions:
  finbar_prim: {0.0: 4, 2.0: 18, 3.0: 170}
  compend_prim: {0: 6, 2: 2, 3: 185}
  disc_sex_prim: {0: 10, 1: 42, 2: 13, 3: 128}
  incl_edu: {0.0: 92, 1.0: 34, 2.0: 19, 3.0: 27, 4.0: 16}


In [43]:
# Drop 'gini_index' and 'gov_edu_spend' due to high missingness:
#   - 'gini_index' is 62.7% missing (would drop our sample to just 72 countries)
#   - 'gov_edu_spend' is 19.7% missing (would cost us an extra 29 countries)
# By excluding both, we preserve a much larger panel of 165 countries.

clustering_metrics = [
    "gdp_pc_ppp", "urban_pct", "youth_dependency_ratio", "lfp_gender_gap",
    "finbar_prim", "compend_prim", "disc_sex_prim", "incl_edu"
]

df_final = df_master.dropna(subset=clustering_metrics).reset_index(drop=True)

import os
os.makedirs("processed", exist_ok=True)
df_final.to_csv("processed/cleaned_global_edu_policy.csv", index=False)

print(f"Final dataset: {df_final.shape[0]} countries × {df_final.shape[1]} columns")
print(f"Feature matrix: {len(clustering_metrics)} variables, 0 missing values")
print()
print(df_final[clustering_metrics].describe().round(2))

Final dataset: 165 countries × 14 columns
Feature matrix: 8 variables, 0 missing values

       gdp_pc_ppp  urban_pct  youth_dependency_ratio  lfp_gender_gap  \
count      165.00     165.00                  165.00          165.00   
mean     28095.70      60.41                    0.44           19.31   
std      29312.85      21.46                    0.21           13.74   
min       1104.78      14.74                    0.16           -2.33   
25%       6647.50      43.28                    0.26            9.53   
50%      17545.62      61.95                    0.38           14.80   
75%      41562.15      76.33                    0.61           25.96   
max     146919.15     100.00                    1.01           65.07   

       finbar_prim  compend_prim  disc_sex_prim  incl_edu  
count       165.00        165.00         165.00    165.00  
mean          2.86          2.90           2.44      1.14  
std           0.48          0.52           0.93      1.38  
min           0.00    